# ScratchML Example Use Cases and scikit-learn Comparisons

This notebook shows how to use the models in `scratchml` on small synthetic datasets and, when available, compares them to equivalent `scikit-learn` models.

The notebook is designed to run even if `scikit-learn` is not installed. In that case, the ScratchML examples still work and the comparison cells print a helpful message.

In [1]:
import numpy as np
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'scratchml').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scratchml import (
    KMeans,
    KNeighborsClassifier,
    KNeighborsRegressor,
    LinearRegression,
    LogisticRegression,
    MinMaxScaler,
    RandomForestClassifier,
    RandomForestRegressor,
    StandardScaler,
    SoftmaxRegression,
    accuracy_score,
    mean_squared_error,
    train_test_split,
)

try:
    from sklearn.cluster import KMeans as SklearnKMeans
    from sklearn.ensemble import RandomForestClassifier as SklearnRandomForestClassifier
    from sklearn.ensemble import RandomForestRegressor as SklearnRandomForestRegressor
    from sklearn.linear_model import LinearRegression as SklearnLinearRegression
    from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
    from sklearn.linear_model import SGDClassifier
    from sklearn.metrics import adjusted_rand_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

print(f"scikit-learn available: {SKLEARN_AVAILABLE}")

scikit-learn available: True


## Helper dataset generators

These keep the notebook self-contained so the examples do not depend on external datasets.

In [2]:
def make_regression_data(n_samples=300, noise=0.15, random_state=42):
    rng = np.random.default_rng(random_state)
    X = rng.normal(size=(n_samples, 3))
    weights = np.array([2.5, -1.2, 0.8])
    y = X @ weights + 1.5 + rng.normal(scale=noise, size=n_samples)
    return X, y


def make_binary_classification_data(n_samples=300, random_state=42):
    rng = np.random.default_rng(random_state)
    X = rng.normal(size=(n_samples, 2))
    margin = 1.8 * X[:, 0] - 1.1 * X[:, 1] + 0.2
    y = (margin > 0).astype(int)
    return X, y


def make_multiclass_data(n_samples=360, random_state=42):
    rng = np.random.default_rng(random_state)
    centers = np.array([[-3.0, -1.0], [0.0, 3.0], [3.0, -1.0]])
    X_parts = []
    y_parts = []
    for label, center in enumerate(centers):
        X_parts.append(rng.normal(loc=center, scale=0.8, size=(n_samples // 3, 2)))
        y_parts.append(np.full(n_samples // 3, label))
    return np.vstack(X_parts), np.concatenate(y_parts)


def adjusted_rand_index(labels_true, labels_pred):
    labels_true = np.asarray(labels_true)
    labels_pred = np.asarray(labels_pred)
    n = len(labels_true)
    classes_true, inverse_true = np.unique(labels_true, return_inverse=True)
    classes_pred, inverse_pred = np.unique(labels_pred, return_inverse=True)
    contingency = np.zeros((len(classes_true), len(classes_pred)), dtype=int)
    for i in range(n):
        contingency[inverse_true[i], inverse_pred[i]] += 1

    def comb2(values):
        values = np.asarray(values, dtype=float)
        return np.sum(values * (values - 1) / 2.0)

    sum_comb_c = comb2(contingency)
    sum_comb_rows = comb2(contingency.sum(axis=1))
    sum_comb_cols = comb2(contingency.sum(axis=0))
    total_pairs = n * (n - 1) / 2.0
    expected_index = (sum_comb_rows * sum_comb_cols) / total_pairs if total_pairs else 0.0
    max_index = 0.5 * (sum_comb_rows + sum_comb_cols)
    denominator = max_index - expected_index
    if denominator == 0:
        return 1.0
    return float((sum_comb_c - expected_index) / denominator)

## 1. Preprocessing with ScratchML scalers

Preprocessing matters most for distance-based models like KNN and optimization-based models like logistic or softmax regression. ScratchML includes a small preprocessing module so a typical workflow can stay inside the library.

In [3]:
raw_X = np.array([
    [1.0, 100.0],
    [2.0, 130.0],
    [3.0, 160.0],
    [4.0, 190.0],
])

standard_scaler = StandardScaler().fit(raw_X)
minmax_scaler = MinMaxScaler(feature_range=(0.0, 1.0)).fit(raw_X)

X_standard = np.array(standard_scaler.transform(raw_X))
X_minmax = np.array(minmax_scaler.transform(raw_X))

print('Original data:')
print(raw_X)
print('\nStandardScaler output:')
print(np.round(X_standard, 3))
print('\nMinMaxScaler output:')
print(np.round(X_minmax, 3))

Original data:
[[  1. 100.]
 [  2. 130.]
 [  3. 160.]
 [  4. 190.]]

StandardScaler output:
[[-1.342 -1.342]
 [-0.447 -0.447]
 [ 0.447  0.447]
 [ 1.342  1.342]]

MinMaxScaler output:
[[0.    0.   ]
 [0.333 0.333]
 [0.667 0.667]
 [1.    1.   ]]


In [4]:
X, y = make_binary_classification_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(n_iters=2000, lr=0.1, random_state=42)
logreg.fit(X_train_scaled, y_train)
scaled_preds = logreg.predict(X_test_scaled)

print('LogisticRegression accuracy after scaling:', round(accuracy_score(y_test, scaled_preds), 4))

LogisticRegression accuracy after scaling: 0.9667


## 2. Linear regression from scratch

A simple regression example with mean squared error on a held-out test set.

In [5]:
X, y = make_regression_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scratch_lr = LinearRegression().fit(X_train, y_train)
scratch_preds = scratch_lr.predict(X_test)
scratch_mse = mean_squared_error(y_test, scratch_preds)

print('ScratchML LinearRegression')
print('Intercept:', round(scratch_lr.intercept_, 4))
print('Coefficients:', np.round(scratch_lr.coef_, 4))
print('Test MSE:', round(scratch_mse, 4))

ScratchML LinearRegression
Intercept: 1.5017
Coefficients: [ 2.5024 -1.1857  0.8006]
Test MSE: 0.0237


In [6]:
if SKLEARN_AVAILABLE:
    sklearn_lr = SklearnLinearRegression().fit(X_train, y_train)
    sklearn_preds = sklearn_lr.predict(X_test)
    sklearn_mse = mean_squared_error(y_test, sklearn_preds)
    print('scikit-learn LinearRegression')
    print('Intercept:', round(float(sklearn_lr.intercept_), 4))
    print('Coefficients:', np.round(sklearn_lr.coef_, 4))
    print('Test MSE:', round(sklearn_mse, 4))
else:
    print('Install scikit-learn to run the direct comparison cell.')

scikit-learn LinearRegression
Intercept: 1.5017
Coefficients: [ 2.5024 -1.1857  0.8006]
Test MSE: 0.0237


## 3. Binary classification with logistic regression

This example uses a linearly separable dataset and compares classification accuracy.

In [7]:
X, y = make_binary_classification_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scratch_logreg = LogisticRegression(n_iters=2000, lr=0.1, random_state=42).fit(X_train, y_train)
scratch_preds = scratch_logreg.predict(X_test)
scratch_acc = accuracy_score(y_test, scratch_preds)

print('ScratchML LogisticRegression accuracy:', round(scratch_acc, 4))
print('Sample probabilities:')
print(np.round(np.array(scratch_logreg.predict_proba(X_test[:5])), 4))

ScratchML LogisticRegression accuracy: 0.9667
Sample probabilities:
[[0.9415 0.0585]
 [0.0024 0.9976]
 [0.0033 0.9967]
 [0.8993 0.1007]
 [0.7254 0.2746]]


In [8]:
if SKLEARN_AVAILABLE:
    sklearn_logreg = SklearnLogisticRegression(max_iter=2000).fit(X_train, y_train)
    sklearn_preds = sklearn_logreg.predict(X_test)
    sklearn_acc = accuracy_score(y_test, sklearn_preds)
    print('scikit-learn LogisticRegression accuracy:', round(sklearn_acc, 4))
else:
    print('Install scikit-learn to compare against sklearn LogisticRegression.')

scikit-learn LogisticRegression accuracy: 0.9667


## 4. Multiclass classification with softmax regression

ScratchML includes a multiclass softmax classifier. For comparison, the cell below uses an SGD-based multinomial classifier from scikit-learn.

In [9]:
X, y = make_multiclass_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scratch_softmax = SoftmaxRegression(n_iters=2500, lr=0.05, random_state=42).fit(X_train, y_train)
scratch_preds = scratch_softmax.predict(X_test)
scratch_acc = accuracy_score(y_test, scratch_preds)

print('ScratchML SoftmaxRegression accuracy:', round(scratch_acc, 4))

ScratchML SoftmaxRegression accuracy: 1.0


In [10]:
if SKLEARN_AVAILABLE:
    sklearn_softmax = SGDClassifier(loss='log_loss', penalty=None, max_iter=3000, tol=1e-4, random_state=42)
    sklearn_softmax.fit(X_train, y_train)
    sklearn_preds = sklearn_softmax.predict(X_test)
    sklearn_acc = accuracy_score(y_test, sklearn_preds)
    print('scikit-learn multinomial-style classifier accuracy:', round(sklearn_acc, 4))
else:
    print('Install scikit-learn to compare against a sklearn multiclass baseline.')

scikit-learn multinomial-style classifier accuracy: 1.0


## 5. Ensemble methods

Random forests are a good example of where the library feels more complete than a single-model demo.

In [11]:
X_reg, y_reg = make_regression_data(n_samples=400, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, random_state=7)

scratch_forest_reg = RandomForestRegressor(n_estimators=25, max_depth=5, max_features=2, random_state=7)
scratch_forest_reg.fit(X_train, y_train)
scratch_mse = mean_squared_error(y_test, scratch_forest_reg.predict(X_test))

print('ScratchML RandomForestRegressor MSE:', round(scratch_mse, 4))

ScratchML RandomForestRegressor MSE: 0.5073


In [12]:
X_clf, y_clf = make_binary_classification_data(n_samples=400, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, random_state=7)

scratch_forest_clf = RandomForestClassifier(n_estimators=25, max_depth=5, max_features=1, random_state=7)
scratch_forest_clf.fit(X_train, y_train)
scratch_acc = accuracy_score(y_test, scratch_forest_clf.predict(X_test))

print('ScratchML RandomForestClassifier accuracy:', round(scratch_acc, 4))

ScratchML RandomForestClassifier accuracy: 0.9625


In [13]:
print('The next cell compares ScratchML and scikit-learn random forests on separate regression and classification datasets.')

The next cell compares ScratchML and scikit-learn random forests on separate regression and classification datasets.


In [14]:
if SKLEARN_AVAILABLE:
    X_reg, y_reg = make_regression_data(n_samples=400, random_state=7)
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, random_state=7)
    sklearn_forest_reg = SklearnRandomForestRegressor(n_estimators=25, max_depth=5, max_features=2, random_state=7)
    sklearn_forest_reg.fit(X_train_r, y_train_r)
    sklearn_mse = mean_squared_error(y_test_r, sklearn_forest_reg.predict(X_test_r))
    print('scikit-learn RandomForestRegressor MSE:', round(sklearn_mse, 4))

    X_clf, y_clf = make_binary_classification_data(n_samples=400, random_state=7)
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, random_state=7)
    sklearn_forest_clf = SklearnRandomForestClassifier(n_estimators=25, max_depth=5, max_features=1, random_state=7)
    sklearn_forest_clf.fit(X_train_c, y_train_c)
    sklearn_acc = accuracy_score(y_test_c, sklearn_forest_clf.predict(X_test_c))
    print('scikit-learn RandomForestClassifier accuracy:', round(sklearn_acc, 4))
else:
    print('Install scikit-learn to compare the random forests.')

scikit-learn RandomForestRegressor MSE: 0.4061
scikit-learn RandomForestClassifier accuracy: 0.95


## 6. Clustering with KMeans

Cluster labels are arbitrary, so adjusted Rand index is a better comparison metric than raw label equality.

In [15]:
X, y = make_multiclass_data()

scratch_kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
scratch_labels = scratch_kmeans.predict(X)
scratch_ari = adjusted_rand_index(y, scratch_labels)

print('ScratchML KMeans inertia:', round(scratch_kmeans.inertia_, 2))
print('ScratchML KMeans adjusted Rand index:', round(scratch_ari, 4))

ScratchML KMeans inertia: 435.73
ScratchML KMeans adjusted Rand index: 0.9917


In [16]:
if SKLEARN_AVAILABLE:
    sklearn_kmeans = SklearnKMeans(n_clusters=3, n_init=10, random_state=42)
    sklearn_labels = sklearn_kmeans.fit_predict(X)
    sklearn_ari = adjusted_rand_score(y, sklearn_labels)
    print('scikit-learn KMeans inertia:', round(float(sklearn_kmeans.inertia_), 2))
    print('scikit-learn KMeans adjusted Rand index:', round(float(sklearn_ari), 4))
else:
    print('Install scikit-learn to run the KMeans comparison.')

scikit-learn KMeans inertia: 435.73
scikit-learn KMeans adjusted Rand index: 0.9917


## 7. Quick API examples

A few short examples showing the library’s scikit-learn-inspired fit/predict style.

In [17]:
reg = KNeighborsRegressor(n_neighbors=2).fit([[0.0], [1.0], [2.0], [3.0]], [0.0, 1.0, 2.0, 3.0])
clf = KNeighborsClassifier(n_neighbors=3).fit([[0.0], [1.0], [2.0], [3.0]], [0, 0, 1, 1])

print('KNeighborsRegressor prediction for 2.5:', reg.predict([2.5]))
print('KNeighborsClassifier prediction for 2.5:', clf.predict([2.5]))

KNeighborsRegressor prediction for 2.5: 2.5
KNeighborsClassifier prediction for 2.5: 1
